# Feature Engineering
**Purpose:** Build the features used for clustering, and keep validation fields separate.

**Input:** `data/cleaned/delhi_clean.csv`

**Outputs:** `data/features/delhi_features.csv`, `data/features/delhi_context.csv`

## 1. Load cleaned data

In [1]:
from pathlib import Path
import pandas as pd, numpy as np

BASE = Path.cwd().parent
df = pd.read_csv(BASE / "data" / "cleaned" / "delhi_clean.csv", parse_dates=["Date"], index_col="Date")
print(df.shape)       

(1195, 16)


## 2. Clustering features
9 pollutants, PM2.5/PM10 ratio (capped at 1.0), temperature, rain, wind speed, and wind direction as sine and cosine. NOx and Toluene are excluded.

In [2]:
feat = pd.DataFrame(index=df.index)
for c in ["PM2.5", "PM10", "NO", "NO2", "NH3", "SO2", "O3", "CO", "Benzene"]:
    feat[c] = df[c]

feat["PM_ratio"] = df["PM2.5"] / df["PM10"]          # fine vs coarse particles
feat["PM_ratio"] = feat["PM_ratio"].clip(upper=1.0)   # ratio can't physically exceed 1
feat["temp"] = df["temperature_2m_mean"]
feat["rain"] = df["precipitation_sum"]
feat["wind_speed"] = df["wind_speed_10m_max"]

rad = np.deg2rad(df["wind_direction_10m_dominant"])   # angle -> two coordinates
feat["wind_dir_sin"] = np.sin(rad)
feat["wind_dir_cos"] = np.cos(rad)

print(feat.shape, feat.isna().sum().sum())

(1195, 15) 0


## 3. Context variables (validation only, never used for clustering)
AQI, year, month, weekday, and approximate flags for Diwali, stubble season, monsoon and lockdown. Dates are approximate.

In [3]:
ctx = pd.DataFrame(index=df.index)
ctx["AQI"] = df["AQI"]
ctx["NOx"] = df["NOx"]
ctx["Toluene"] = df["Toluene"]
ctx["year"] = df.index.year
ctx["month"] = df.index.month
ctx["dayofweek"] = df.index.dayofweek
ctx["is_weekend"] = ctx["dayofweek"] >= 5

diwali = pd.to_datetime(["2017-10-19", "2018-11-07", "2019-10-27"])
ctx["diwali_window"] = [any(abs((d - x).days) <= 3 for x in diwali) for d in df.index]

ctx["stubble_season"] = ((ctx["month"] == 10) & (df.index.day >= 15)) | (ctx["month"] == 11)
ctx["monsoon"] = ctx["month"].isin([7, 8, 9])
ctx["lockdown"] = (df.index >= "2020-03-25") & (df.index <= "2020-05-31")
print(ctx[["diwali_window", "stubble_season", "monsoon", "lockdown"]].sum())

diwali_window      21
stubble_season    141
monsoon           190
lockdown           68
dtype: int64


## 4. PM ratio by month and save
The ratio is highest in November to January (fine-particle smog) and lowest in April to September (coarse particles).

In [4]:
print(feat["PM_ratio"].groupby(feat.index.month).mean().round(2))

(BASE / "data" / "features").mkdir(parents=True, exist_ok=True)
feat.to_csv(BASE / "data" / "features" / "delhi_features.csv")
ctx.to_csv(BASE / "data" / "features" / "delhi_context.csv")

Date
1     0.65
2     0.53
3     0.46
4     0.39
5     0.39
6     0.36
7     0.40
8     0.40
9     0.43
10    0.53
11    0.63
12    0.62
Name: PM_ratio, dtype: float64


## 5. Check: days where PM2.5 exceeds PM10
4 days between 1.01 and 1.15, capped at 1.0.

In [5]:
raw_ratio = df["PM2.5"] / df["PM10"]       # in notebook 04, where df is loaded
bad = raw_ratio > 1
print(bad.sum())
print(df.loc[bad, ["PM2.5", "PM10"]].assign(ratio=raw_ratio[bad]).round(2))

4
             PM2.5    PM10  ratio
Date                             
2017-01-19  227.68  225.03   1.01
2017-01-20  217.55  207.03   1.05
2017-02-23  182.29  158.96   1.15
2017-10-20  426.52  387.13   1.10
